# ClarIA — Pipeline completo: N estados de cuenta → Tablas DB

```
PDFs  →  pdfplumber (texto)  →  Claude Haiku (JSON)  →  tarjetas / transacciones / msi
```

**Resuelve:**
1. Múltiples estados de cuenta (misma tarjeta o distintas)
2. Identifica el último estado por tarjeta → saldo actual correcto
3. Asigna IDs a tarjetas, transacciones y planes MSI
4. Deduplica transacciones que aparecen en varios periodos
5. Exporta a CSV y genera SQL listo para producción

## 0. Setup

In [ ]:
%pip install anthropic pdfplumber python-dotenv pandas --quiet

In [ ]:
import pdfplumber
import anthropic
import json
import os
import re
import glob
from pathlib import Path
from datetime import datetime
from dotenv import load_dotenv
import pandas as pd

load_dotenv(Path('..') / '.env')
client = anthropic.Anthropic(api_key=os.getenv('ANTHROPIC_API_KEY'))

print('✅ Listo')
print(f'   API key: {os.getenv("ANTHROPIC_API_KEY", "NO ENCONTRADA")[:16]}...')

## 1. Configuración

In [ ]:
# Directorio con todos los PDFs de estados de cuenta
PDF_DIR   = 'bbva'
MODEL     = 'claude-haiku-4-5-20251001'
MAX_CHARS = 12000   # caracteres que se pasan a Claude
MAX_TOKENS_RESP = 4096

# Colores por defecto para tarjetas BBVA
CARD_COLORS = [
    ('#002C7A', '#0058C8'),
    ('#880000', '#CC1E00'),
    ('#003F6B', '#006EA8'),
    ('#1A4731', '#276749'),
    ('#2D1B69', '#5B2D8E'),
]

## 2. Prompt y funciones de extracción

In [ ]:
# ── PROMPT (declarado por el usuario) ─────────────────────────────────────────
PROMPT_TEMPLATE = """Eres un extractor de datos de estados de cuenta de tarjeta de crédito BBVA México.
Del siguiente texto extrae TODOS los campos y responde SOLO con JSON válido, sin markdown, sin texto extra.
Estructura exacta a devolver:
{{
  "nombre_tarjeta": "nombre del producto (ej: BBVA Dorada, Azul, Platino, Clásica)",
  "banco": "BBVA",
  "last4": "últimos 4 dígitos del número de tarjeta (string)",
  "limite": 0,
  "saldo_usado": 0,
  "saldo_pagar": 0,
  "dia_corte": 15,
  "dia_pago": 10,
  "periodo": "DD-MMM-YYYY al DD-MMM-YYYY",
  "transacciones_msi": [
    {{
      "fecha_operacion": "DD-MMM-YYYY",
      "descripcion": "descripción tal como aparece",
      "monto_original": 0,
      "saldo_pendiente": 0,
      "pago_requerido": 0,
      "num_pago_actual": 4,
      "num_pagos_total": 12,
      "tasa_interes": 0.00
    }}
  ],
  "transacciones_regulares": [
    {{
      "fecha_operacion": "DD-MMM-YYYY",
      "fecha_cargo": "DD-MMM-YYYY",
      "descripcion": "descripción tal como aparece",
      "monto": 0,
      "tipo": "abono"
    }}
  ]
}}

─── REGLAS GENERALES ───────────────────────────────────────────────────────────
- limite: línea de crédito total en MXN (número, sin comas ni $)
- saldo_usado: saldo al corte o total a pagar en MXN (número, sin comas)
- dia_corte / dia_pago: día del mes como entero (1-31)
- Si un campo no aparece, usa null
- Todas las fechas en formato DD-MMM-YYYY (ej: 10-feb-2026)

─── REGLAS: transacciones_msi ──────────────────────────────────────────────────
Sección: "COMPRAS Y CARGOS DIFERIDOS A MESES SIN INTERESES"
- monto_original: columna "Monto original" (número, sin comas ni $)
- saldo_pendiente: columna "Saldo pendiente" (número, sin comas ni $)
- pago_requerido: columna "Pago requerido" (número, sin comas ni $)
- num_pago_actual / num_pagos_total: extraídos de "Núm. de pago" (ej: "4 de 12" → 4 y 12)
- tasa_interes: columna "Tasa de interés aplicable" como decimal (ej: "0.00%" → 0.00)

─── REGLAS: transacciones_regulares ────────────────────────────────────────────
Sección: "CARGOS, COMPRAS Y ABONOS REGULARES (NO A MESES)"
- monto: número positivo si es cargo (+), negativo si es abono (-)
  · Los abonos aparecen como "- $X" o negativos → monto negativo (ej: -15000.00)
  · Los cargos aparecen como "+ $X" o sin signo → monto positivo (ej: 3411.00)
- tipo: "cargo" si monto > 0, "abono" si monto < 0
- descripcion: solo la primera línea del movimiento (sin el desglose de IVA/intereses)

Texto del estado de cuenta:
{texto}"""

In [ ]:
MESES_ES = {
    'ene':1,'feb':2,'mar':3,'abr':4,'may':5,'jun':6,
    'jul':7,'ago':8,'sep':9,'oct':10,'nov':11,'dic':12
}

def parse_fecha_bbva(s: str) -> datetime:
    """'09-mar-2026' → datetime(2026, 3, 9)"""
    d, m, y = s.strip().lower().split('-')
    return datetime(int(y), MESES_ES[m], int(d))

def fecha_corte_del_periodo(periodo: str) -> datetime:
    """'10-feb-2026 al 09-mar-2026' → datetime del día de corte"""
    return parse_fecha_bbva(periodo.split(' al ')[-1].strip())

def extraer_texto_pdf(path: str) -> str:
    with pdfplumber.open(path) as pdf:
        return '\n\n'.join(
            f'--- Página {i+1} ---\n{p.extract_text() or ""}'
            for i, p in enumerate(pdf.pages)
        )

def extraer_info_tarjeta(texto: str) -> dict:
    prompt = PROMPT_TEMPLATE.format(texto=texto[:MAX_CHARS])
    response = client.messages.create(
        model=MODEL, max_tokens=MAX_TOKENS_RESP,
        messages=[{'role': 'user', 'content': prompt}]
    )
    raw = re.sub(r'^```(?:json)?\s*|\s*```$', '', response.content[0].text.strip())
    return json.loads(raw), response.usage

print('✅ Funciones listas')

## 3. Procesar todos los PDFs

Cada PDF se parsea una vez. El resultado se guarda en `raw_estados`.

In [ ]:
pdfs = sorted(glob.glob(f'{PDF_DIR}/*.pdf'))
print(f'📂 {len(pdfs)} PDFs encontrados en ./{PDF_DIR}/\n')

raw_estados = []   # lista de dicts: info extraída + metadata del archivo
errores     = []

for path in pdfs:
    nombre = Path(path).name
    print(f'  ⏳ {nombre}', end='  ')
    try:
        texto  = extraer_texto_pdf(path)
        info, usage = extraer_info_tarjeta(texto)
        info['_archivo'] = nombre
        info['_tokens']  = usage.input_tokens + usage.output_tokens
        raw_estados.append(info)
        print(f'✅  {info.get("nombre_tarjeta")} ****{info.get("last4")}  |  {info.get("periodo")}  |  {usage.input_tokens+usage.output_tokens} tokens')
    except Exception as e:
        errores.append({'archivo': nombre, 'error': str(e)})
        print(f'❌  {e}')

print(f'\n──────────────────────────────────────────────────')
print(f'✅ {len(raw_estados)} procesados   ❌ {len(errores)} errores')
print(f'💰 Tokens totales: {sum(e["_tokens"] for e in raw_estados):,}')

## 4. Identificar tarjetas únicas y último estado por tarjeta

Agrupa por `last4`. Ordena por `fecha_corte` (parseada del campo `periodo`).
El último estado de cuenta de cada tarjeta define: `saldo_usado`, `saldo_pagar`, `limite`.

In [ ]:
from collections import defaultdict

# Enriquecer cada estado con fecha_corte parseada
for e in raw_estados:
    try:
        e['_fecha_corte'] = fecha_corte_del_periodo(e['periodo'])
    except Exception:
        e['_fecha_corte'] = datetime(2000, 1, 1)  # fallback

# Agrupar por last4
por_tarjeta = defaultdict(list)
for e in raw_estados:
    por_tarjeta[e['last4']].append(e)

# Ordenar cada grupo por fecha_corte
for last4 in por_tarjeta:
    por_tarjeta[last4].sort(key=lambda x: x['_fecha_corte'])

print(f'Tarjetas únicas encontradas: {len(por_tarjeta)}\n')
for last4, estados in por_tarjeta.items():
    ultimo = estados[-1]
    print(f'  ****{last4}  {ultimo["nombre_tarjeta"]:25s}'
          f'  {len(estados)} estado(s)'
          f'  último: {ultimo["periodo"]}')
    print(f'           saldo_usado=${ultimo["saldo_usado"]:,.2f}   '
          f'límite=${ultimo.get("limite") or 0:,.2f}')

## 5. Tabla `tarjetas`

In [ ]:
# Asignar tarjeta_id secuencial (1, 2, 3…)
tarjeta_id_map = {}   # last4 → tarjeta_id
filas_tarjetas = []

for idx, (last4, estados) in enumerate(sorted(por_tarjeta.items()), start=1):
    tarjeta_id_map[last4] = idx
    ultimo = estados[-1]                    # ← último estado de cuenta
    c1, c2 = CARD_COLORS[(idx - 1) % len(CARD_COLORS)]
    filas_tarjetas.append({
        'tarjeta_id'  : idx,
        'nombre'      : ultimo['nombre_tarjeta'],
        'banco'       : ultimo['banco'],
        'last4'       : last4,
        'limite'      : ultimo.get('limite') or 0,
        'saldo_usado' : ultimo.get('saldo_usado') or 0,
        'saldo_pagar' : ultimo.get('saldo_pagar') or 0,
        'dia_corte'   : ultimo.get('dia_corte') or 15,
        'dia_pago'    : ultimo.get('dia_pago')  or 10,
        'color_inicio': c1,
        'color_fin'   : c2,
        'num_estados' : len(estados),
        'periodo_inicio': estados[0]['periodo'],
        'periodo_fin'   : estados[-1]['periodo'],
    })

df_tarjetas = pd.DataFrame(filas_tarjetas)
print(f'🏦 Tabla tarjetas — {len(df_tarjetas)} fila(s)\n')
df_tarjetas[['tarjeta_id','nombre','last4','limite','saldo_usado','dia_corte','dia_pago','num_estados']]

## 6. Tabla `transacciones_regulares`

**Deduplicación**: una misma transacción puede aparecer en varios estados de cuenta
(el movimiento de un periodo aparece como pendiente en el siguiente).
Clave de deduplicación: `(tarjeta_id, fecha_operacion, descripcion_normalizada, monto)`.

In [ ]:
def normalizar(s: str) -> str:
    """Normaliza descripción para deduplicar (quita sufijos de tarjeta digital)."""
    # Quita el sufijo '; Tarjeta Digital ***XXXX'
    s = re.sub(r'\s*;\s*Tarjeta Digital \*+\d+', '', s, flags=re.IGNORECASE)
    return s.strip().upper()

vistos    = set()   # clave de dedup
filas_tx  = []
tx_id_seq = 0

# Iterar en orden cronológico (todos los estados de todas las tarjetas)
estados_ordenados = sorted(raw_estados, key=lambda x: x['_fecha_corte'])

for estado in estados_ordenados:
    last4      = estado['last4']
    tarjeta_id = tarjeta_id_map[last4]
    periodo    = estado['periodo']

    for tx in (estado.get('transacciones_regulares') or []):
        # Clave de dedup
        clave = (
            tarjeta_id,
            tx.get('fecha_operacion', ''),
            normalizar(tx.get('descripcion', '')),
            tx.get('monto', 0)
        )
        if clave in vistos:
            continue
        vistos.add(clave)

        tx_id_seq += 1
        filas_tx.append({
            'id'             : tx_id_seq,
            'tarjeta_id'     : tarjeta_id,
            'fecha_operacion': tx.get('fecha_operacion'),
            'fecha_cargo'    : tx.get('fecha_cargo'),
            'descripcion'    : tx.get('descripcion'),
            'monto'          : tx.get('monto', 0),
            'tipo'           : tx.get('tipo'),
            'periodo'        : periodo,
            'via'            : 'pdf',
        })

df_tx = pd.DataFrame(filas_tx)
print(f'💳 Tabla transacciones — {len(df_tx)} registros únicos\n')
print(f'   Cargos : {(df_tx["tipo"]=="cargo").sum()}')
print(f'   Abonos : {(df_tx["tipo"]=="abono").sum()}')
if len(df_tx):
    print(f'   Fecha más antigua : {df_tx["fecha_operacion"].min()}')
    print(f'   Fecha más reciente: {df_tx["fecha_operacion"].max()}')
df_tx.head(10)

## 7. Tabla `compras_msi`

**Deduplicación MSI**: el mismo plan aparece en cada estado con `num_pago_actual` distinto.
Clave: `(tarjeta_id, descripcion_normalizada, monto_original)` → guardar la entrada
con el `num_pago_actual` más alto (más reciente).

In [ ]:
msi_map   = {}   # clave → dict con los datos más recientes

for estado in estados_ordenados:          # ya ordenados cronológicamente
    last4      = estado['last4']
    tarjeta_id = tarjeta_id_map[last4]

    for msi in (estado.get('transacciones_msi') or []):
        clave = (
            tarjeta_id,
            normalizar(msi.get('descripcion', '')),
            msi.get('monto_original', 0)
        )
        # Guardar siempre el más reciente (el de mayor num_pago_actual)
        prev = msi_map.get(clave)
        if prev is None or (msi.get('num_pago_actual') or 0) > (prev.get('num_pago_actual') or 0):
            msi_map[clave] = {
                'tarjeta_id'    : tarjeta_id,
                'descripcion'   : msi.get('descripcion'),
                'fecha_operacion': msi.get('fecha_operacion'),
                'monto_original': msi.get('monto_original', 0),
                'saldo_pendiente': msi.get('saldo_pendiente', 0),
                'pago_requerido': msi.get('pago_requerido', 0),
                'num_pago_actual': msi.get('num_pago_actual', 1),
                'num_pagos_total': msi.get('num_pagos_total', 1),
                'tasa_interes'  : msi.get('tasa_interes', 0),
                'estado_fuente' : estado['periodo'],
            }

msi_id_seq   = 0
filas_msi    = []
for clave, datos in msi_map.items():
    msi_id_seq += 1
    filas_msi.append({'id': msi_id_seq, **datos})

df_msi = pd.DataFrame(filas_msi) if filas_msi else pd.DataFrame()
print(f'🔒 Tabla compras_msi — {len(df_msi)} planes únicos\n')
if len(df_msi):
    cols = ['id','tarjeta_id','descripcion','monto_original','saldo_pendiente','num_pago_actual','num_pagos_total']
    print(df_msi[cols].to_string(index=False))

## 8. Verificación y resumen

In [ ]:
print('═' * 60)
print('RESUMEN FINAL')
print('═' * 60)
print(f'  Tarjetas únicas  : {len(df_tarjetas)}')
print(f'  Transacciones    : {len(df_tx)}')
print(f'    Cargos         : {(df_tx["tipo"]=="cargo").sum() if len(df_tx) else 0}')
print(f'    Abonos         : {(df_tx["tipo"]=="abono").sum() if len(df_tx) else 0}')
print(f'  Planes MSI       : {len(df_msi)}')

if len(df_tx):
    total_cargos = df_tx.loc[df_tx['tipo']=='cargo', 'monto'].sum()
    total_abonos = df_tx.loc[df_tx['tipo']=='abono', 'monto'].abs().sum()
    print(f'\n  Suma cargos      : ${total_cargos:>12,.2f}')
    print(f'  Suma abonos      : ${total_abonos:>12,.2f}')

print('\n── Por tarjeta ──────────────────────────────────────')
for _, row in df_tarjetas.iterrows():
    tid = row['tarjeta_id']
    n_tx = len(df_tx[df_tx['tarjeta_id']==tid]) if len(df_tx) else 0
    n_msi = len(df_msi[df_msi['tarjeta_id']==tid]) if len(df_msi) else 0
    print(f'  [{tid}] {row["nombre"]:25s} ****{row["last4"]}  '
          f'{n_tx} tx  {n_msi} MSI  saldo=${row["saldo_usado"]:,.2f}')

## 9. Export a CSV

In [ ]:
OUT_DIR = Path('output')
OUT_DIR.mkdir(exist_ok=True)

df_tarjetas.to_csv(OUT_DIR / 'tarjetas.csv', index=False)
df_tx.to_csv(OUT_DIR / 'transacciones.csv', index=False)
if len(df_msi):
    df_msi.to_csv(OUT_DIR / 'compras_msi.csv', index=False)

print('✅ CSVs guardados en ./output/')
print(f'   tarjetas.csv       ({len(df_tarjetas)} filas)')
print(f'   transacciones.csv  ({len(df_tx)} filas)')
print(f'   compras_msi.csv    ({len(df_msi)} filas)')

## 10. Generar SQL para producción

Listo para ejecutar en la DB de Render. Revisa primero los CSVs.

In [ ]:
def sql_val(v):
    if v is None or (isinstance(v, float) and pd.isna(v)):
        return 'NULL'
    if isinstance(v, str):
        return "'" + v.replace("'", "''") + "'"
    return str(v)

# ── INSERT tarjetas ────────────────────────────────────────────
lineas_t = ['-- Tarjetas',
            'INSERT INTO tarjetas (id,usuario_id,nombre,banco,last4,limite,saldo_usado,dia_corte,dia_pago,color_inicio,color_fin,activo)',
            'VALUES']
for i, row in df_tarjetas.iterrows():
    sep = ',' if i < len(df_tarjetas)-1 else ';'
    lineas_t.append(
        f"  ({row['tarjeta_id']},1,{sql_val(row['nombre'])},{sql_val(row['banco'])},"
        f"{sql_val(str(row['last4']))},{row['limite']},{row['saldo_usado']},"
        f"{row['dia_corte']},{row['dia_pago']},{sql_val(row['color_inicio'])},{sql_val(row['color_fin'])},TRUE){sep}"
    )

sql_tarjetas = '\n'.join(lineas_t)
print(sql_tarjetas)
(OUT_DIR / 'insert_tarjetas.sql').write_text(sql_tarjetas)
print('\n✅ Guardado en output/insert_tarjetas.sql')

In [ ]:
# ── INSERT transacciones ───────────────────────────────────────
# Convierte formato de fecha BBVA → ISO para PostgreSQL
def bbva_a_iso(s):
    if not s:
        return None
    try:
        return parse_fecha_bbva(s).strftime('%Y-%m-%d')
    except Exception:
        return None

lineas = [
    '-- Transacciones',
    'INSERT INTO transacciones',
    '  (usuario_id,tarjeta_id,fecha_operacion,fecha_cargo,descripcion,monto_mxn,tipo,periodo,via)',
    'VALUES'
]
for i, row in df_tx.iterrows():
    sep  = ',' if i < len(df_tx)-1 else ';'
    fop  = sql_val(bbva_a_iso(row['fecha_operacion']))
    fcar = sql_val(bbva_a_iso(row['fecha_cargo']))
    lineas.append(
        f"  (1,{row['tarjeta_id']},{fop},{fcar},{sql_val(row['descripcion'])},"
        f"{abs(row['monto'])},{sql_val(row['tipo'])},{sql_val(row['periodo'])},'pdf'){sep}"
    )

sql_tx = '\n'.join(lineas)
(OUT_DIR / 'insert_transacciones.sql').write_text(sql_tx)
print(f'✅ {len(df_tx)} transacciones → output/insert_transacciones.sql')
print('\nPrimeras 5 líneas:')
print('\n'.join(sql_tx.split('\n')[:8]))

In [ ]:
# ── INSERT compras_msi ─────────────────────────────────────────
if len(df_msi):
    lineas = [
        '-- Compras MSI',
        'INSERT INTO compras_msi',
        '  (usuario_id,tarjeta_id,descripcion,monto_total,saldo_pendiente,total_pagos,pagos_hechos,cuota_mensual,fecha_inicio)',
        'VALUES'
    ]
    for i, row in df_msi.iterrows():
        sep    = ',' if i < len(df_msi)-1 else ';'
        cuota  = round(row['monto_original'] / max(row['num_pagos_total'], 1), 2)
        finicio = sql_val(bbva_a_iso(row['fecha_operacion']))
        lineas.append(
            f"  (1,{row['tarjeta_id']},{sql_val(row['descripcion'])},"
            f"{row['monto_original']},{row['saldo_pendiente']},"
            f"{row['num_pagos_total']},{row['num_pago_actual']},{cuota},{finicio}){sep}"
        )
    sql_msi = '\n'.join(lineas)
    (OUT_DIR / 'insert_msi.sql').write_text(sql_msi)
    print(f'✅ {len(df_msi)} planes MSI → output/insert_msi.sql')
    print()
    print(sql_msi)
else:
    print('Sin planes MSI encontrados.')